In [1]:
import pandas as pd
import numpy as np

In [2]:
# existing databases
name_db = pd.read_csv('name_db.csv')[['cleanName']].dropna().reset_index(drop=True)
name_db['cleanName'] = name_db['cleanName'].str.title()

product_db = pd.read_csv('product_db.csv', encoding='latin1')[['name', 'price']].dropna().reset_index(drop=True)
product_db['price'] = product_db['price'].str[1:].str.replace(',', '').astype(float).astype(int)

store_db = pd.read_csv('store_db.csv')[['company_name', 'city', 'state', 'zip_code']].dropna().reset_index(drop=True)
store_db['zip_code'] = store_db['zip_code'].astype(int).astype('O')

street_db = pd.read_csv('street_db.csv')[['FullStreetName']].dropna().reset_index(drop=True)
street_db = street_db[~street_db['FullStreetName'].str.startswith('UNNAMED')].reset_index(drop=True)
street_db['FullStreetName'] = street_db['FullStreetName'].str.title()
for i in range(len(street_db)):
    street = street_db.loc[i, 'FullStreetName']
    if street[0].isdigit():
        street_db.loc[i, 'FullStreetName'] = street.replace(street.split()[0], street.split()[0].lower())

zipcode_db = pd.read_csv('zipcode_db.csv')[['state_abbr', 'zipcode', 'city']].dropna()
zipcode_db = zipcode_db[zipcode_db['zipcode'].apply(len) == 5]
zipcode_db = zipcode_db[~zipcode_db['zipcode'].str.contains('[A-Z]')]
zipcode_db = zipcode_db[~zipcode_db['city'].str.startswith('Zcta')].reset_index(drop=True)
zipcode_db['city'] = zipcode_db['city'].str.title()

In [3]:
# store
store_df = pd.DataFrame(columns=['ID', 'storeName', 'address', 'city', 'ZIP','state', 'phoneNumber'])
store_N = 100
for i in range(store_N):
    rand_store = np.random.randint(0, len(store_db))
    rand_street = np.random.randint(0, len(street_db))
    
    store_name = store_db.loc[rand_store, 'company_name'].replace(",", "")
    address = street_db.loc[rand_street, 'FullStreetName']
    city = store_db.loc[rand_store, 'city']
    zipcode = int(store_db.loc[rand_store, 'zip_code'])
    state = store_db.loc[rand_store, 'state']
    
    phone_number = str(np.random.randint(1000000000, 9999999999))
    phone_number = phone_number[:3] + ' ' + phone_number[3:6] + ' ' + phone_number[6:]
    
    store_df.loc[i] = [i + 1, store_name, address, city, zipcode, state, phone_number]
    
with open('store.txt', 'w') as f:
    for index, row in store_df.iterrows():
        f.write(', '.join(map(str, row)) + '\n')

In [4]:
# customer
customer_df = pd.DataFrame(columns=['ID', 'name', 'birthDate', 'address', 'city', 'ZIP', 'state', 'phoneNumber'])
customer_N = 1000
for i in range(customer_N):
    rand_name = np.random.randint(0, len(name_db))
    rand_street = np.random.randint(0, len(street_db))
    rand_zipcode = np.random.randint(0, len(zipcode_db))
    
    name = name_db.loc[rand_name, 'cleanName']
    
    year = '{:02d}'.format(np.random.randint(1900, 2011))
    month = '{:02d}'.format(np.random.randint(1, 13))
    if month == '2':
        day = '{:02d}'.format(np.random.randint(1, 30))
    else:
        day = '{:02d}'.format(np.random.randint(1, 32))
    birth_date = '/'.join([year, month, day])
    
    address = street_db.loc[rand_street, 'FullStreetName']
    city = zipcode_db.loc[rand_zipcode, 'city']
    zipcode = int(zipcode_db.loc[rand_zipcode, 'zipcode'])
    state = zipcode_db.loc[rand_zipcode, 'state_abbr']
    
    phone_number = str(np.random.randint(1000000000, 9999999999))
    phone_number = phone_number[:3] + ' ' + phone_number[3:6] + ' ' + phone_number[6:]
    
    customer_df.loc[i] = [i + 1, name, birth_date, address, city, zipcode, state, phone_number]

with open('customer.txt', 'w') as f:
    for index, row in customer_df.iterrows():
        f.write(', '.join(map(str, row)) + '\n')

In [5]:
# sales
# select store IDs and customer IDs from existing
# at least one sale for every customer and every store
sales_df = pd.DataFrame(columns=['ID', 'date', 'time', 'storeID', 'customerID'])
sales_N = 2000
for i in range(sales_N):
    year = '{:02d}'.format(np.random.randint(1950, 2025))
    month = '{:02d}'.format(np.random.randint(1, 13))
    if month == '2':
        day = '{:02d}'.format(np.random.randint(1, 30))
    else:
        day = '{:02d}'.format(np.random.randint(1, 32))
    date = '/'.join([year, month, day])
    
    hour = '{:02d}'.format(np.random.randint(0, 25))
    minute = '{:02d}'.format(np.random.randint(0, 60))
    sec = '{:02d}'.format(np.random.randint(0, 60))
    time = ':'.join([hour, minute, sec])
    
    storeID = np.random.randint(1, store_N + 1)
    customerID = np.random.randint(1, customer_N + 1)
    # assume always more customers than stores
    if i < customer_N:
        if i < store_N:
            if (sales_df['storeID'] == storeID).any(): # if already in
                while (sales_df['storeID'] == storeID).any(): # keep generating until get new
                    storeID = np.random.randint(1, store_N + 1)
        # same logic here
        if (sales_df['customerID'] == customerID).any():
            while (sales_df['customerID'] == customerID).any():
                customerID = np.random.randint(1, customer_N + 1)
        customerID = np.random.randint(1, customer_N + 1)
    
    sales_df.loc[i] = [i + 1, date, time, storeID, customerID]

with open('sales.txt', 'w') as f:
    for index, row in sales_df.iterrows():
        f.write(', '.join(map(str, row)) + '\n')

In [6]:
# product
product_df = pd.DataFrame(columns=['ID', 'description', 'price'])
product_N = 100
for i in range(product_N):
    rand_product = np.random.randint(0, len(product_db))
    desc = product_db.loc[rand_product, 'name']
    price = '{:.2f}'.format(product_db.loc[rand_product, 'price'])
    
    product_df.loc[i] = [i + 1, desc, price]
    
with open('product.txt', 'w') as f:
    for index, row in product_df.iterrows():
        f.write(', '.join(map(str, row)) + '\n')

In [7]:
# lineItem
# reference existing sales and product
# every sale consists of one or more line items (ie appears at least once)
line_item_df = pd.DataFrame(columns=['ID', 'salesID', 'productID', 'quantity'])
line_item_N = 4000
for i in range(line_item_N):
    salesID = np.random.randint(1, sales_N + 1)
    if i < sales_N:
        if (line_item_df['salesID'] == salesID).any(): # if already in
            while (line_item_df['salesID'] == salesID).any(): # keep generating until get new
                salesID = np.random.randint(1, sales_N + 1)

    productID = np.random.randint(1, product_N + 1)
    quantity = np.random.randint(1, 21)
    
    line_item_df.loc[i] = [i + 1, salesID, productID, quantity]

with open('lineItem.txt', 'w') as f:
    for index, row in line_item_df.iterrows():
        f.write(', '.join(map(str, row)) + '\n')